In [1]:
import sys

sys.path.append("..")
from src.data import laad_seizoenen

df = laad_seizoenen("../data/seizoenen")
df.shape

(1637, 28)

In [2]:
per_seizoen = df.groupby("seizoen")
per_seizoen.size(), per_seizoen["FTR"].apply(lambda s: (s == "H").mean())

(seizoen
 21-22    306
 22-23    306
 23-24    312
 24-25    312
 25-26    311
 26-27     90
 dtype: int64,
 seizoen
 21-22    0.418301
 22-23    0.441176
 23-24    0.445513
 24-25    0.439103
 25-26    0.405145
 26-27    0.511111
 Name: FTR, dtype: float64)

In [3]:
df["FTR"].value_counts(normalize=True)

FTR
H    0.434331
A    0.320098
D    0.245571
Name: proportion, dtype: float64

In [4]:
df.groupby(["seizoen", "HomeTeam"]).size().unstack(0)

seizoen,21-22,22-23,23-24,24-25,25-26,26-27
HomeTeam,,,,,,
Anderlecht,17.0,17.0,20.0,20.0,20.0,5.0
Antwerp,17.0,17.0,20.0,20.0,20.0,5.0
Beerschot VA,17.0,NaN,NaN,18.0,NaN,NaN
Beveren,NaN,NaN,NaN,NaN,NaN,5.0
Cercle Brugge,17.0,17.0,20.0,18.0,18.0,5.0
Charleroi,17.0,17.0,18.0,20.0,20.0,5.0
Club Brugge,17.0,17.0,20.0,20.0,20.0,5.0
Dender,NaN,NaN,NaN,20.0,17.0,NaN
Eupen,17.0,17.0,18.0,NaN,NaN,NaN


In [5]:
marge = 1 / df["B365H"] + 1 / df["B365D"] + 1 / df["B365A"] - 1
marge.describe()

count    1631.000000
mean        0.073946
std         0.014416
min         0.031933
25%         0.063492
50%         0.070330
75%         0.084656
max         0.123810
dtype: float64

In [6]:
df[df["B365H"].isna()]

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,AC,HY,AY,HR,AR,B365H,B365D,B365A,seizoen,thuiswinst
265,B1,2022-03-06,12:30,Antwerp,Beerschot VA,2,1,H,0.0,1.0,...,3.0,2.0,7.0,0.0,1.0,NaN,NaN,NaN,21-22,1
1013,B1,2024-10-26,15:00,Charleroi,Oud-Heverlee Leuven,0,2,A,0.0,2.0,...,5.0,0.0,2.0,0.0,0.0,NaN,NaN,NaN,24-25,0
1199,B1,2025-04-24,19:30,Club Brugge,St. Gilloise,0,1,A,0.0,1.0,...,2.0,1.0,3.0,0.0,0.0,NaN,NaN,NaN,24-25,0
1544,B1,2026-05-24,17:30,Club Brugge,Gent,5,0,H,3.0,0.0,...,8.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,25-26,1
1545,B1,2026-05-24,17:30,St Truiden,Mechelen,3,0,H,2.0,0.0,...,2.0,2.0,2.0,0.0,0.0,NaN,NaN,NaN,25-26,1
1546,B1,2026-05-24,17:30,St. Gilloise,Anderlecht,5,1,H,4.0,0.0,...,1.0,2.0,2.0,0.0,1.0,NaN,NaN,NaN,25-26,1


In [7]:
import pandas as pd

thuis = pd.DataFrame(
    {
        "wedstrijd": df.index,
        "Date": df["Date"],
        "ploeg": df["HomeTeam"],
        "voor": df["FTHG"],
        "tegen": df["FTAG"],
        "schoten": df["HS"],
        "punten": df["FTR"].map({"H": 3, "D": 1, "A": 0}),
    }
)
uit = pd.DataFrame(
    {
        "wedstrijd": df.index,
        "Date": df["Date"],
        "ploeg": df["AwayTeam"],
        "voor": df["FTAG"],
        "tegen": df["FTHG"],
        "schoten": df["AS"],
        "punten": df["FTR"].map({"H": 0, "D": 1, "A": 3}),
    }
)
per_ploeg = pd.concat([thuis, uit]).sort_values(["ploeg", "Date"])
per_ploeg.head(10)

,wedstrijd,Date,ploeg,voor,tegen,schoten,punten
6,6,2021-07-25,Anderlecht,1,3,9.0,0
13,13,2021-07-31,Anderlecht,1,1,10.0,1
25,25,2021-08-08,Anderlecht,3,0,19.0,3
34,34,2021-08-15,Anderlecht,2,1,17.0,3
50,50,2021-08-29,Anderlecht,0,1,16.0,0
55,55,2021-09-12,Anderlecht,7,2,15.0,3
68,68,2021-09-19,Anderlecht,1,0,12.0,3
71,71,2021-09-23,Anderlecht,1,1,13.0,1
78,78,2021-09-26,Anderlecht,2,2,18.0,1
86,86,2021-10-03,Anderlecht,1,1,18.0,1


In [8]:
len(per_ploeg)

3274

In [9]:
import numpy as np

# oudste eerst, vorige match laatst: 20x1, 5x2, 4x3, 1x5 = 30 wedstrijden
GEWICHTEN = np.array([1] * 20 + [2] * 5 + [3] * 4 + [5])


def gewogen(reeks):
    w = GEWICHTEN[-len(reeks) :]  # minder dan 30 matchen? neem de laatste gewichten
    ok = ~np.isnan(reeks)  # lege waarden tellen niet mee
    if not ok.any():
        return np.nan
    return np.average(reeks[ok], weights=w[ok])


per_ploeg["vorm_punten"] = per_ploeg.groupby("ploeg")["punten"].transform(
    lambda s: s.shift(1).rolling(30, min_periods=1).apply(gewogen, raw=True)
)
per_ploeg.head(10)

,wedstrijd,Date,ploeg,voor,tegen,schoten,punten,vorm_punten
6,6,2021-07-25,Anderlecht,1,3,9.0,0,NaN
13,13,2021-07-31,Anderlecht,1,1,10.0,1,0.000000
25,25,2021-08-08,Anderlecht,3,0,19.0,3,0.625000
34,34,2021-08-15,Anderlecht,2,1,17.0,3,1.636364
50,50,2021-08-29,Anderlecht,0,1,16.0,0,1.928571
55,55,2021-09-12,Anderlecht,7,2,15.0,3,1.235294
68,68,2021-09-19,Anderlecht,1,0,12.0,3,1.894737
71,71,2021-09-23,Anderlecht,1,1,13.0,1,2.095238
78,78,2021-09-26,Anderlecht,2,2,18.0,1,1.739130
86,86,2021-10-03,Anderlecht,1,1,18.0,1,1.600000


In [10]:
for kolom in ["punten", "voor", "tegen", "schoten"]:
    per_ploeg[f"vorm_{kolom}"] = per_ploeg.groupby("ploeg")[kolom].transform(
        lambda s: s.shift(1).rolling(30, min_periods=1).apply(gewogen, raw=True)
    )
per_ploeg["aantal_vorige"] = per_ploeg.groupby("ploeg").cumcount().clip(upper=30)

vorm = per_ploeg.set_index(["wedstrijd", "ploeg"])
vormkolommen = ["vorm_punten", "vorm_voor", "vorm_tegen", "vorm_schoten", "aantal_vorige"]
thuisvorm = vorm.loc[list(zip(df.index, df["HomeTeam"], strict=True)), vormkolommen].to_numpy()
uitvorm = vorm.loc[list(zip(df.index, df["AwayTeam"], strict=True)), vormkolommen].to_numpy()
for i, k in enumerate(vormkolommen):
    df[f"verschil_{k}"] = thuisvorm[:, i] - uitvorm[:, i]
df.filter(like="verschil_").describe()

,verschil_vorm_punten,verschil_vorm_voor,verschil_vorm_tegen,verschil_vorm_schoten,verschil_aantal_vorige
count,1621.000000,1621.000000,1621.000000,1619.000000,1637.000000
mean,-0.023116,-0.014829,0.016363,-0.171949,0.004887
std,0.682934,0.618953,0.606928,2.891910,5.576319
min,-3.000000,-3.000000,-2.875000,-13.000000,-30.000000
25%,-0.489362,-0.426652,-0.361702,-2.148936,0.000000
50%,-0.040000,-0.021277,0.021277,-0.191489,0.000000
75%,0.425532,0.382979,0.423077,1.826707,0.000000
max,2.625000,2.000000,3.000000,10.404255,30.000000


In [11]:
df["thuiswinst"] = (df["FTR"] == "H").astype(int)
df["vorm_groep"] = pd.qcut(
    df["verschil_vorm_punten"],
    5,
    labels=["veel slechter", "slechter", "gelijk", "beter", "veel beter"],
)
df.groupby("vorm_groep", observed=True)["thuiswinst"].agg(["mean", "size"])

,mean,size
vorm_groep,,
veel slechter,0.261538,325
slechter,0.327160,324
gelijk,0.393846,325
beter,0.529412,323
veel beter,0.666667,324


In [12]:
from src.features import maak_features

ruw = laad_seizoenen("../data/seizoenen")
maak_features(ruw).filter(like="verschil_").describe()

,verschil_vorm_punten,verschil_vorm_voor,verschil_vorm_tegen,verschil_vorm_schoten,verschil_aantal_vorige
count,1621.000000,1621.000000,1621.000000,1619.000000,1637.000000
mean,-0.023116,-0.014829,0.016363,-0.171949,0.004887
std,0.682934,0.618953,0.606928,2.891910,5.576319
min,-3.000000,-3.000000,-2.875000,-13.000000,-30.000000
25%,-0.489362,-0.426652,-0.361702,-2.148936,0.000000
50%,-0.040000,-0.021277,0.021277,-0.191489,0.000000
75%,0.425532,0.382979,0.423077,1.826707,0.000000
max,2.625000,2.000000,3.000000,10.404255,30.000000


In [13]:
from src.data import laad_seizoenen

ruw = laad_seizoenen("../data/seizoenen")
print(ruw["thuiswinst"].value_counts(dropna=False))
print(ruw.groupby("FTR", dropna=False)["thuiswinst"].first())

thuiswinst
0    926
1    711
Name: count, dtype: int64
FTR
A    0
D    0
H    1
Name: thuiswinst, dtype: int64


In [14]:
from src.features import maak_features

df = maak_features(ruw)
lege = df["verschil_vorm_punten"].isna()
print(lege.sum(), "wedstrijden met lege vorm")
print(df.loc[lege, ["seizoen", "HomeTeam", "AwayTeam"]].groupby("seizoen").size())

16 wedstrijden met lege vorm
seizoen
21-22    9
22-23    1
23-24    1
24-25    1
25-26    1
26-27    3
dtype: int64


In [15]:
from src import schema
from src.data import laad_seizoenen

ruw = laad_seizoenen("../data/seizoenen")
schema.GESPEELD.validate(ruw, lazy=True)
print("schema ok")
kapot = ruw.copy()
kapot.loc[0, "FTR"] = "X"
schema.GESPEELD.validate(kapot, lazy=True)

schema ok


SchemaErrors: {
    "DATA": {
        "DATAFRAME_CHECK": [
            {
                "schema": "gespeelde wedstrijden",
                "column": "FTR",
                "check": "isin(['H', 'D', 'A'])",
                "error": "Column 'FTR' failed element-wise validator number 0: isin(['H', 'D', 'A']) failure cases: X"
            },
            {
                "schema": "gespeelde wedstrijden",
                "column": "gespeelde wedstrijden",
                "check": "FTR klopt niet",
                "error": "DataFrameSchema 'gespeelde wedstrijden' failed element-wise validator number 2: <Check <lambda>: FTR klopt niet> failure cases: B1, 2021-07-23 00:00:00, 19:45, Standard, Genk, 1, 1, X, 0.0, 0.0, D, 17.0, 17.0, 5.0, 5.0, 11.0, 8.0, 7.0, 3.0, 1.0, 2.0, 0.0, 0.0, 2.75, 3.5, 2.37, 21-22, 0"
            }
        ]
    }
}